In [ ]:
'''
문제 생성. 테마 랜덤 배치

재영 (1501~3000)

In [ ]:
import os
api_key = os.environ.get("GEMINI_API_KEY")

In [16]:
import pandas as pd
import json
import time
import os
import random

# 최신 구글 공식 SDK 임포트
from google import genai
from google.genai import types

# ==========================================
# 1. SSAFY 테마 리스트 정의
# ==========================================
SSAFY_THEMES = [
    "구성원: 20대 후반~30대 초반의 교육생(약 20~30명이 1개 분반), 출석/태도를 관리하는 담당프로, 코칭을 담당하는 강사, SSAFY를 창시한 '싸버지'라는 교육프로.",
    "공간: 서울, 대전, 광주, 구미, 부산 캠퍼스. 내부에 교육장, 복도, 화장실, 라운지(회의 공간), 식당, 엘리베이터 존재.",
    "시간/일정: 매일 아침 9시 출근 ~ 저녁 18시 실습 및 수강 종료. 점심시간은 12시~13시.",
    "특이사항(서울캠퍼스): 10층(도시락/샌드위치/샐러드 중 택1), 20층(한식/양식 중 택1) 식당 이용. 4층과 지하 1층은 교육생 출입 금지.",
    "서울 캠퍼스에는 엘리베이터가 3개 있으며, 계단이 2개 있음. 전체 층이 20층으로 높아서, 가끔은 엘리베이터보다 계단으로 가는 편이 빠를 수도 있음.",
    "서울 캠퍼스 근처의 카페는 총 5군데가 있음. 거리가 가까운 순으로 '바나프레소, 메가커피, 커피빈, (길 건너편) 테라로사, 스타벅스'가 있음.",
    "서울 캠퍼스의 식사 시간은 12~14시인데, 3, 2, 7, 6, 5층이 각각 1시간씩, 15분 간격으로 밥을 먹으며 매달 로테이션됨. (예: 5월은 3층 12:00, 2층 12:15... 다음 달은 15분씩 밀림)",
    "서울 캠퍼스 10층 식당의 메뉴들은 수량이 한정되어 있어 12:30~13:00 사이에 특정 메뉴가 소진되는 경우가 종종 있음.",
    "각 분반의 반장 및 C.A는 교육생들이 지원/선출하며, 이들에게는 매달 예산이 지급되어 분반에 필요한 물건이나 간식을 자율적으로 구매함.",
    "각 분반의 문화에 따라 자주, 혹은 가끔 교육생들끼리 일과 후에 캠퍼스 근처 식당에서 회식을 하기도 함.",
    "캠퍼스 내 규정을 위반한 교육생 적발 시 1차 경고, 2차 제재를 가함. 'SSAFY 사무국'의 '박준우 교육프로'가 제재를 총괄함.",
    "규정 위반 사례: 학생증 미패용, 교육 중 게임, 강의장 밖 슬리퍼 착용, 지정된 식사시간 외 점심식사 등.",
    "'월간 베스트' 제도: 매달 분반에서 교육태도가 좋은 1명을 추천받아, 가장 많이 추천받은 교육생 명단을 공개하고 기프티콘을 지급함.",
    "교육지원금: 매달 100만원 지급. 1회 지각 시마다 약 5만원이 차감됨.",
    "트랙: Web(Python), Web(Java), Embedded, Mobile, Embedded Robot, Data 등 다양한 과정 존재.",
    "비전공반/전공반: 전공에 상관없이 지원 가능하며, 비전공반은 인문, 자연과학, 공학 등 다양한 전공자로 구성됨.",
    "기수: 1년 커리큘럼. 6개월 단위로 약 900명이 선발됨. 2026년 1월~12월은 15기, 2026년 7월부터는 16기가 교육 받음.",
    "중도퇴소: 자진퇴소, 출결 등 규정위반 퇴소, 조기 취업에 의한 퇴소 등이 있음."
    "식당에서 음식을 먹기 위해서는 학생증이 있어야 함.",
    "몇 개월에 한 번씩 소프트웨어 역량평가가 있음. IM형, A형, B형, C형 시험이 있으며, 비전공자는 1학기(6개월) 내에 IM형을 취득해야 하며 전공자는 A형을 취득해야 함. 취득하지 못 할 시 수료 조건에 미달되어 2학기로 진학할 수 없음.",
    "소프트웨어 역량평가 A형 취득자를 대상으로 토요일마다 4시간씩, 8주간 B형 특강을 진행함.",
    "특정 과목 커리큘럼이 종료될 때마다 소규모 관통 프로젝트를 각 분반에서 2명씩 팀을 이루어 하루 동안 진행함.",
    "평일 오전에 그날의 식단을 MM(Mattermost SNS 앱 이름의 줄임말)에 자동으로 올려주는 '싸단 알리미'가 있음.",
    "매달 한 번씩 오후 4시~6시 사이에 강당에서 취업특강을 진행함.",
    "각 강의실에는 천장에 에어컨, 뒷편에 공기청정기와 화이트보드가 있음.",
    "각 층에는 호수의 1자리 수가 1~5인 강의실들이 있음. 예를 들어 6층에는 601호 ~ 605호가 있음.",
    "각 층에는 남자화장실, 여자화장실이 층의 한 쪽 끝에 함께 있음.",
    "서울캠퍼스의 엘리베이터 3대는 출근시간, 점심시간, 퇴근시간에 이용자가 많아서, 해당 시간대에는 캠퍼스 소속 관리요원들이 이용자들의 대기줄을 통제하여 각 엘리베이터별 탑승 인원을 조절함.",
    "서울캠퍼스의 10층 식당에는 좌석의 종류가 2인석 4곳, 4인석 4곳, 6인석 4곳, 나머지 1인석은 창가에 대략 8곳 배치되어 있는데, 각 좌석들의 수는 정확하지 않으므로 비슷한 개수로 변경 가능.",
    "SSAFY 내에서 교육생끼리 소통하거나 SSAFY 사무국에서 중요한 공지를 모든 교육생에게 전달할 때에는 Mattermost(줄여서 MM으로 부름) 앱을 이용함.",
    "소프트웨어 역량평가 B형을 취득한 교육생에게는 100,000P의 장학포인트를 매달 지급하며, 이 장학포인트는 이후 SSAFY 전용 삼성 제품 마켓에서 현금처럼 이용이 가능함",
    "1학기에 진행하는 과목평가 중에는 '일타싸피'라는 과목이 있음. 일타싸피는 포켓볼 게임의 각 공의 위치 정보가 주어졌을 때, 흰 공을 치는 각도와 세기를 결정하는 알고리즘을 작성하는 시험.",
    "1학기에 진행하는 과목평가 중에는 '배틀싸피'라는 과목이 있음. 배틀싸피는 아군 탱크와 적군 탱크들과 장애물들의 위치 정보가 주어졌을 때, 적군 탱크를 모두 격침하도록 아군 탱크를 이동하고 포탄을 발사하는 알고리즘을 작성하는 시험."
]

# ==========================================
# 2. Gemini API 설정 함수
# ==========================================
def setup_gemini(api_key):
    client = genai.Client(api_key=api_key)
    
    safety_settings = [
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
    ]

    # system_instruction을 제거하여 Gemma 모델의 500 에러 원천 차단
    config = types.GenerateContentConfig(
        temperature=0.7,
        response_mime_type="application/json",
        safety_settings=safety_settings
    )
    
    return client, config

# ==========================================
# 3. 문제 변환 요청 함수
# ==========================================
def transform_problem_with_gemini(client, config, original_question, max_retries=10):
    # 매 요청마다 5개의 랜덤 테마를 추출
    selected_themes = random.sample(SSAFY_THEMES, 4)
    themes_text = "\n".join([f"- {theme}" for theme in selected_themes])

    # 메인 프롬프트에 모든 지시사항 통합
    prompt = f"""
You are an expert in creating algorithmic coding test problems. 
Your task is to translate the following English algorithm problem into Korean and change its story/theme.

[CRITICAL RULES]
1. EXACT LOGIC: The mathematical and logical constraints (time/space complexity, variable ranges, core algorithmic rules) MUST remain 100% identical to the original.
2. NATURAL KOREAN: The translation must be perfectly natural Korean without any awkward translation tone (번역투). the names of the characters must be natural korean people's like '민수' or '지원' if they are not specified in the constraints.
3. THEME ADAPTATION: Below are a few specific concepts regarding an academy called 'SSAFY'. Weave these concepts into the story naturally. 
   **HOWEVER, if the provided SSAFY concepts do not logically fit the algorithm's constraints, DO NOT force them.** Instead, create a normal, everyday Korean story theme.
4. TITLE: Generate a concise, catchy, and appropriate algorithm problem title in Korean.

[SSAFY Theme Concepts to use (Optional)]
{themes_text}

[Original Problem (Includes description, input, output, examples)]
{original_question}

Provide the output STRICTLY in the following JSON format without markdown tags:
{{
    "generated_title": "(Korean) Concise and fitting algorithm problem title",
    "generated_question": "(Korean) The transformed main problem description",
    "generated_exp_input": "(Korean) Explanation of the input format and constraints",
    "generated_exp_output": "(Korean) Explanation of the expected output",
    "generated_exp_example": "(Korean) Step-by-step explanation of the example cases"
}}
"""
    
    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model="gemma-4-31b-it",
                contents=prompt,
                config=config
            )
            
            text = response.text.strip()
            md_marker = "`" * 3
            if text.startswith(f'{md_marker}json'):
                text = text.replace(f'{md_marker}json', '').replace(md_marker, '').strip()
            elif text.startswith(md_marker):
                text = text.replace(md_marker, '').strip()

            result_dict = json.loads(text)
            
            return (
                result_dict.get("generated_title", ""),
                result_dict.get("generated_question", ""),
                result_dict.get("generated_exp_input", ""),
                result_dict.get("generated_exp_output", ""),
                result_dict.get("generated_exp_example", "")
            )
            
        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg or "503" in error_msg:
                wait_time = (10 * attempt) + random.uniform(1, 3)
                print(f"  -> ⚠️ 서버 지연 감지! {wait_time:.1f}초 대기 후 재시도... ({attempt}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"  -> ⚠️ 파싱 등 기타 에러: {e}. 5초 대기 후 재시도... ({attempt}/{max_retries})")
                time.sleep(5)
                
    return None, None, None, None, None


# %% [Cell 2: 메인 실행 로직]
# ==========================================
# 사용자 설정 (Variables)
# ==========================================
# api_key 변수는 이미 선언되어 있다고 가정합니다. (예: api_key = "AIzaSy...")

INPUT_FILE = "validated_problems.csv"
OUTPUT_FILE = "translated_problems-2.csv"

# [분산 처리용 범위 설정] (테이블 기준 상단 첫 번째 = 1)
START_IDX = 1251       # 시작 문제 순서
END_IDX = 1400       # 끝 문제 순서

# ==========================================
# 메인 로직 시작
# ==========================================
print("🚀 [Step 1] 원본 문제 데이터 로드 및 열 구조 준비...")
overall_start_time = time.time()

if os.path.exists(OUTPUT_FILE):
    print(f"📂 기존 작업 파일({OUTPUT_FILE})을 이어서 불러옵니다.")
    df = pd.read_csv(OUTPUT_FILE)
else:
    print(f"📄 원본 파일({INPUT_FILE})을 새로 불러옵니다.")
    df = pd.read_csv(INPUT_FILE)

# 신규 생성할 5개의 열 초기화
new_cols = ['generated_title', 'generated_question', 'generated_exp_input', 'generated_exp_output', 'generated_exp_example']
for col in new_cols:
    if col not in df.columns:
        df[col] = None

# question 열 바로 오른쪽에 신규 열들이 오도록 데이터프레임 열 순서 재배치
cols = list(df.columns)
for col in new_cols:
    cols.remove(col)
q_idx = cols.index('question')
# question 열 바로 뒤에 title, 그 다음 question 등 순서대로 삽입
cols = cols[:q_idx+1] + new_cols + cols[q_idx+1:]
df = df[cols]

# 작업 대상 범위 자르기
start_idx = max(0, START_IDX - 1)
end_idx = min(len(df), END_IDX)
target_problems = df.iloc[start_idx:end_idx]

print(f"✅ 총 {len(df)}문제 중 {START_IDX}번째 ~ {end_idx}번째 문제(총 {len(target_problems)}개) 변환 작업을 시작합니다.\n")

client, config = setup_gemini(api_key)
processed_count = 0

# 5. 문제 생성 루프
for table_idx, (index, row) in enumerate(target_problems.iterrows(), start=START_IDX):
    prob_id = row['id']
    original_question = row['question']

    # 이전의 어떤 solution때문에 id가 깨진 것 -> 숫자 아님 -> 스킵
    if not str(prob_id).isdigit():
        print(f"⏭ [{table_idx}번째] Problem {prob_id} 은(는) 숫자가 아니라서 건너뜁니다.")
        continue
    
    # 이미 생성된 결과가 있다면 스킵
    if pd.notnull(row['generated_question']) and str(row['generated_question']).strip() != "":
        print(f"⏭ [{table_idx}번째] Problem {prob_id} 은(는) 이미 변환이 완료되어 건너뜁니다.")
        continue
        
    print(f"\n⏳ [{table_idx}번째] Problem {prob_id} 테마 변환 및 번역 중...")
    step_start_time = time.time()
    
    # LLM 호출
    gen_title, gen_q, gen_in, gen_out, gen_ex = transform_problem_with_gemini(client, config, original_question)
    
    step_end_time = time.time()
    elapsed_time = step_end_time - step_start_time
    
    if gen_q:
        df.at[index, 'generated_title'] = gen_title
        df.at[index, 'generated_question'] = gen_q
        df.at[index, 'generated_exp_input'] = gen_in
        df.at[index, 'generated_exp_output'] = gen_out
        df.at[index, 'generated_exp_example'] = gen_ex
        
        # 미리보기용 텍스트 가공 (줄바꿈 제거 및 50글자 제한)
        preview_text = gen_q.replace('\n', ' ')
        preview_text = preview_text[:50] + "..." if len(preview_text) > 50 else preview_text
        
        print(f"  -> ✅ 변환 완료! (소요 시간: {elapsed_time:.1f}초)")
        print(f"  -> 🔍 [미리보기] {gen_title} | {preview_text}")
        processed_count += 1
    else:
        print(f"  -> ❌ 최대 재시도 초과. 변환에 실패했습니다. (소요 시간: {elapsed_time:.1f}초)")
    
    # 실시간 저장
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    
    time.sleep(6.5)

overall_end_time = time.time()
total_elapsed = overall_end_time - overall_start_time
minutes, seconds = divmod(total_elapsed, 60)

print(f"\n🎉 작업 완료! 이번 실행에서 총 {processed_count}개의 문제가 변환되었습니다.")
print(f"⏱️ 총 누적 소요 시간: {int(minutes)}분 {seconds:.1f}초")
print(f"📁 결과 저장 파일: {OUTPUT_FILE}")

🚀 [Step 1] 원본 문제 데이터 로드 및 열 구조 준비...
📂 기존 작업 파일(translated_problems-2.csv)을 이어서 불러옵니다.


/tmp/ipykernel_52322/3893284814.py:167: DtypeWarning: Columns (0: id, 1: generated_title, 2: generated_question, 3: generated_exp_input, 4: generated_exp_output, 5: generated_exp_example, 6: count_cases, 7: count_solutions, 8: Expected Auxiliary Space, 9: starter_code, 10: picture_num, 11: Unnamed: 21, 12: Unnamed: 22, 13: Unnamed: 23, 14: Unnamed: 25, 15: Unnamed: 26, 16: Unnamed: 30, 17: Unnamed: 32, 18: Unnamed: 36, 19: Unnamed: 50, 20: Unnamed: 54, 21: Unnamed: 56, 22: Unnamed: 60, 23: Unnamed: 80, 24: Unnamed: 84, 25: Unnamed: 92, 26: Unnamed: 102, 27: Unnamed: 110, 28: Unnamed: 114, 29: Unnamed: 120, 30: Unnamed: 126, 31: Unnamed: 140, 32: Unnamed: 144, 33: Unnamed: 150, 34: Unnamed: 156, 35: Unnamed: 164, 36: Unnamed: 170, 37: Unnamed: 173) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(OUTPUT_FILE)


✅ 총 4634문제 중 1251번째 ~ 1400번째 문제(총 150개) 변환 작업을 시작합니다.

⏭ [1251번째] Problem 7506 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1252번째] Problem 7511 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1253번째] Problem 7512 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1254번째] Problem 7513 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1255번째] Problem 7519 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1256번째] Problem 7521 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1257번째] Problem 7525 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1258번째] Problem 7530 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1259번째] Problem 7535 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1260번째] Problem 7539 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1261번째] Problem 7540 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1262번째] Problem 7543 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1263번째] Problem 7548 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1264번째] Problem 7566 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1265번째] Problem 7569 은(는) 이미 변환이 완료되어 건너뜁니다.
⏭ [1266번째] Problem 7570 은(는) 이미 변환이 완료되어 건너뜁니다.

⏳ [1267번째] Problem 7577 테마 변환 및 번역 중...
  -> ⚠️ 파싱 등 기타 에러: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}. 5초 대기 후 재시도.

KeyboardInterrupt: 